In [6]:
F83 = GF(83)
P = PolynomialRing(F83, 'x')
x = P.gen()

R = P.quotient(x^4 + 1, 't')
t = R.gen()

In [7]:
A = Matrix(R, 2, 3, [
    23 + 64*t + 11*t^2 + 45*t^3,
     5 + 38*t + 72*t^2 + 19*t^3,
    48 + 12*t + 33*t^2 + 67*t^3,
    76 + 21*t +  4*t^2 + 13*t^3,
    15 + 59*t + 28*t^2 + 50*t^3,
     9 + 42*t + 61*t^2 +  3*t^3,
])

print(A)

[45*t^3 + 11*t^2 + 64*t + 23  19*t^3 + 72*t^2 + 38*t + 5 67*t^3 + 33*t^2 + 12*t + 48]
[ 13*t^3 + 4*t^2 + 21*t + 76 50*t^3 + 28*t^2 + 59*t + 15   3*t^3 + 61*t^2 + 42*t + 9]


In [8]:
# Перетворимо матрицю A над кільцем многочленів у матрицю над полем F83,
# її розміри стануть 8x12

M = Matrix(F83, 8, 12)

for i in range(2):
    for j in range(3):
        for k in range(4):
            p = (A[i, j] * t^k).lift()
            for l in range(4):
                M[4 * i + l, 4 * j + k] = p[l]
            

print(f"M = {M}")
print()
print(f"rank(M) = {M.rank()}")

M_reduced = M.rref()

M = [23 38 72 19  5 64 11 45 48 16 50 71]
[64 23 38 72 38  5 64 11 12 48 16 50]
[11 64 23 38 72 38  5 64 33 12 48 16]
[45 11 64 23 19 72 38  5 67 33 12 48]
[76 70 79 62 15 33 55 24  9 80 22 41]
[21 76 70 79 59 15 33 55 42  9 80 22]
[ 4 21 76 70 28 59 15 33 61 42  9 80]
[13  4 21 76 50 28 59 15  3 61 42  9]

rank(M) = 8


In [9]:
from itertools import product

In [14]:
# Дістаємо позиції вільних змінних, відкидаючи одиничні вектори
unity_vecs = M_reduced.pivots()

free = [j for j in range(M.ncols()) if j not in unity_vecs]

coefs = [[-int(M_reduced[i, j]) for j in free] for i in range(len(unity_vecs))]



def solve(maxnorm: int = 10):
    solutions = list()
    for values in product(range(-maxnorm, maxnorm+1), repeat=len(free)):
        if not any(values): # Відкидаємо одразу нульовий вектор
            continue 
    
        pivot_values = [
            (sum(a*b for a, b in zip(row, values)) + 41) % 83 - 41
            for row in coefs
        ]
    
        if all(abs(c) <= maxnorm for c in pivot_values):
            solution = [0] * M.ncols()
            for j, c in zip(free, values):
                solution[j] = c
            for j, c in zip(unity_vecs, pivot_values):
                solution[j] = c
            solutions.append(solution)
    return solutions

solutions = solve()


In [15]:
if not solutions:
    print("Рішення немає.")
else:
    for solution in solutions:
        z = vector(R, [
            sum(solution[4*j + k] * t^k for k in range(4))
        for j in range(3)
        ])
        assert A * z == vector(R, [0, 0])
        print("Коефіцієнти зі знаками:", solution)
        print("Многочлен:", z)


Рішення немає.


In [16]:
# Якщо підняти допуск до норми = 12, то розвʼязків вже буде багато.
solutions = solve(12)

if not solutions:
    print("Рішення немає.")
else:
    for solution in solutions:
        z = vector(R, [
            sum(solution[4*j + k] * t^k for k in range(4))
        for j in range(3)
        ])
        assert A * z == vector(R, [0, 0])
        print("Коефіцієнти зі знаками:", solution)
        print("Многочлен:", z)

Коефіцієнти зі знаками: [3, -2, 8, -5, 8, 3, 1, -1, -12, 11, -5, -12]
Многочлен: (78*t^3 + 8*t^2 + 81*t + 3, 82*t^3 + t^2 + 3*t + 8, 71*t^3 + 78*t^2 + 11*t + 71)
Коефіцієнти зі знаками: [-5, -3, 2, -8, -1, -8, -3, -1, -12, 12, -11, 5]
Многочлен: (75*t^3 + 2*t^2 + 80*t + 78, 82*t^3 + 80*t^2 + 75*t + 82, 5*t^3 + 72*t^2 + 12*t + 71)
Коефіцієнти зі знаками: [0, 11, 3, 12, -3, -4, 9, -5, -11, -5, 0, -5]
Многочлен: (12*t^3 + 3*t^2 + 11*t, 78*t^3 + 9*t^2 + 79*t + 80, 78*t^3 + 78*t + 72)
Коефіцієнти зі знаками: [2, -8, 5, 3, -3, -1, 1, 8, -11, 5, 12, -12]
Многочлен: (3*t^3 + 5*t^2 + 75*t + 2, 8*t^3 + t^2 + 82*t + 80, 71*t^3 + 12*t^2 + 5*t + 72)
Коефіцієнти зі знаками: [-6, -9, -1, 0, -12, -12, -3, 8, -11, 6, 6, 5]
Многочлен: (82*t^2 + 74*t + 77, 8*t^3 + 80*t^2 + 71*t + 71, 5*t^3 + 6*t^2 + 6*t + 72)
Коефіцієнти зі знаками: [8, -2, 3, 2, 8, -11, 10, -1, -10, -6, -2, 0]
Многочлен: (2*t^3 + 3*t^2 + 81*t + 8, 82*t^3 + 10*t^2 + 72*t + 8, 81*t^2 + 77*t + 73)
Коефіцієнти зі знаками: [-8, 9, 5, -10, -1